# 3. Exploración de Bronze

Propósito: Verificar que los datos de bronze se descargaron correctamente.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: c:\Users\user\Downloads\EP-GDM-G6
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot


In [2]:
from pyspark.sql import SparkSession
from app.utils.spark import SparkClient

# Reutiliza la sesion activa del kernel; si no hay, crea una con la config
# Windows-correcta de SparkClient (rutas nativas Hadoop, memoria del driver).
spark = SparkSession.getActiveSession() or SparkClient().get_session()
bronze_path = "data/bronze"
print("Spark", spark.version)

Spark 4.1.1


In [3]:
manifest = spark.read.parquet(f"{bronze_path}/manifest.parquet")
manifest.show()

+-----------+--------------------+---------+---------+--------------------+
|source_name|         module_name|row_count|file_size|       downloaded_at|
+-----------+--------------------+---------+---------+--------------------+
|       SIAF|        2021-Ingreso|   828167|  9957236|2026-06-19T20:32:...|
|       SIAF|        2022-Ingreso|   774954|  9681483|2026-06-19T20:32:...|
|       SIAF|        2023-Ingreso|   743452|  9280285|2026-06-19T20:32:...|
|       SIAF|        2024-Ingreso|   758116|  9399847|2026-06-19T20:32:...|
|       SIAF|        2025-Ingreso|   878730| 10463637|2026-06-19T20:32:...|
|       SIAF|        2026-Ingreso|   354316|  4608246|2026-06-19T20:32:...|
|   SISMEPRE|rentas_ano_aplica...|       26|     3885|2026-06-19T20:32:...|
|   SISMEPRE|rentas_entidad_es...|    19037|    50883|2026-06-19T20:32:...|
|   SISMEPRE|rentas_esat_estad...|   134170|  1925885|2026-06-19T20:32:...|
|   SISMEPRE|  rentas_estadistica|      233|     3141|2026-06-19T20:32:...|
|   SISMEPRE

In [4]:
import os
files = [f for f in os.listdir(bronze_path) if f.endswith('.parquet') and f != 'manifest.parquet']
print(f"Archivos bronze: {len(files)}")
for f in sorted(files):
    df = spark.read.parquet(f"{bronze_path}/{f}")
    print(f"{f}: {df.count()} filas, {len(df.columns)} columnas")

Archivos bronze: 16
RENAMU-2021.parquet: 1874 filas, 1300 columnas
RENAMU-2022.parquet: 1874 filas, 1368 columnas
RENAMU-2023.parquet: 1891 filas, 1383 columnas
RENAMU-2024.parquet: 1891 filas, 1388 columnas
RENAMU-984-Modulo1963.parquet: 1891 filas, 1388 columnas
SIAF-2021-Ingreso.parquet: 828167 filas, 36 columnas
SIAF-2022-Ingreso.parquet: 774954 filas, 36 columnas
SIAF-2023-Ingreso.parquet: 743452 filas, 36 columnas
SIAF-2024-Ingreso.parquet: 758116 filas, 36 columnas
SISMEPRE-rentas_ano_aplicacion.parquet: 26 filas, 9 columnas
SISMEPRE-rentas_entidad_estado.parquet: 19037 filas, 13 columnas
SISMEPRE-rentas_esat_estadistica_atm.parquet: 134144 filas, 42 columnas
SISMEPRE-rentas_estadistica.parquet: 233 filas, 7 columnas
SISMEPRE-rentas_formulario.parquet: 98 filas, 10 columnas
SISMEPRE-rentas_preguntas.parquet: 836 filas, 15 columnas
SISMEPRE-rentas_respuestas.parquet: 250179 filas, 11 columnas


In [5]:
df = spark.read.parquet(f"{bronze_path}/SIAF-2021-Ingreso.parquet")
df.limit(5).toPandas()

,ANO_DOC,MES_DOC,NIVEL_GOBIERNO,NIVEL_GOBIERNO_NOMBRE,SECTOR,SECTOR_NOMBRE,PLIEGO,PLIEGO_NOMBRE,SEC_EJEC,EJECUTORA,...,SUBGENERICA_NOMBRE,SUBGENERICA_DET,SUBGENERICA_DET_NOMBRE,ESPECIFICA,ESPECIFICA_NOMBRE,ESPECIFICA_DET,ESPECIFICA_DET_NOMBRE,MONTO_PIA,MONTO_PIM,MONTO_RECAUDADO
0,2021,3,M,GOBIERNOS LOCALES,None,None,None,None,301257,150108,...,RENTAS DE LA PROPIEDAD,1,RENTAS DE LA PROPIEDAD FINANCIERA,1,INTERESES,1,INTERESES POR DEPOSITOS DISTINTOS DE RECURSOS ...,0,0,0
1,2021,12,M,GOBIERNOS LOCALES,None,None,None,None,301257,150108,...,MULTAS Y SANCIONES NO TRIBUTARIAS,2,SANCIONES,1,SANCIONES ADMINISTRATIVAS,99,OTRAS SANCIONES,0,0,1300.05
2,2021,10,M,GOBIERNOS LOCALES,None,None,None,None,301257,150108,...,DERECHOS Y TASAS ADMINISTRATIVOS,8,DERECHOS ADMINISTRATIVOS DE TRANSPORTES Y COMU...,1,DERECHOS ADMINISTRATIVOS DE TRANSPORTES Y COMU...,6,ESTACIONAMIENTO DE VEHICULOS,0,0,5321.80
3,2021,8,M,GOBIERNOS LOCALES,None,None,None,None,301257,150108,...,DONACIONES Y TRANSFERENCIAS CORRIENTES,3,DE OTRAS UNIDADES DE GOBIERNO,1,DE OTRAS UNIDADES DE GOBIERNO,1,DEL GOBIERNO NACIONAL,0,0,99915
4,2021,4,M,GOBIERNOS LOCALES,None,None,None,None,301257,150108,...,RENTAS DE LA PROPIEDAD,1,RENTAS DE LA PROPIEDAD FINANCIERA,1,INTERESES,1,INTERESES POR DEPOSITOS DISTINTOS DE RECURSOS ...,0,0,71.58
